In [1]:
import os
import subprocess
import tempfile

In [2]:
class NIST_STS:
    def __init__(self, sts_path="./NIST_sp800-22/nist-sts/sts.exe"):
        self.sts_path = sts_path
        if not os.path.exists(sts_path):
            raise FileNotFoundError(f"sts.exe not found at {sts_path}")

    def write_binary_file(self, bits, filename="data.bin"):
        """
        Convert a string/list of bits (0/1) into a binary file for STS.
        """
        if isinstance(bits, str):
            bits = bits.strip()
        elif isinstance(bits, (list, tuple)):
            bits = ''.join(str(b) for b in bits)
        else:
            raise ValueError("Bits must be a string of '0'/'1' or list of ints.")

        # pad to multiple of 8
        if len(bits) % 8 != 0:
            bits += "0" * (8 - len(bits) % 8)

        b = int(bits, 2).to_bytes(len(bits) // 8, "big")

        filepath = os.path.join(os.path.dirname(self.sts_path), filename)
        with open(filepath, "wb") as f:
            f.write(b)
        return filepath

    def run_tests(self, binfile, seq_length=1000000, num_seq=1):
        """
        Run NIST STS on the given binary file.
        """
        sts_dir = os.path.dirname(self.sts_path)
        input_path = os.path.join(sts_dir, binfile)

        if not os.path.exists(input_path):
            raise FileNotFoundError(f"Binary file not found: {input_path}")

        # Prepare command file for automatic input
        cmd_file = tempfile.NamedTemporaryFile(delete=False, mode="w", dir=sts_dir)
        cmd_file.write(f"{binfile}\n")            # file name
        cmd_file.write(f"{seq_length}\n")         # length of sequence
        cmd_file.write(f"{num_seq}\n")            # number of sequences
        cmd_file.write("1\n")                     # run all tests
        cmd_file.close()

        # Run STS with redirected input
        result = subprocess.run(
            [self.sts_path],
            cwd=sts_dir,
            stdin=open(cmd_file.name),
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True
        )

        os.unlink(cmd_file.name)

        # Read the final report
        report_path = os.path.join(sts_dir, "experiments", "AlgorithmTesting", "finalAnalysisReport.txt")
        report = None
        if os.path.exists(report_path):
            with open(report_path, "r") as f:
                report = f.read()

        return result.stdout, result.stderr, report


In [ ]:
# Example: generate 1 million random bits using Python
import numpy as np

rand_bits = ''.join(str(b) for b in np.random.randint(0, 2, size=1000000))

# Run STS
sts = NIST_STS("C:\\Drive\\GITHUB\\Mani_IITPKD_Research\\Surrogate_TRNG\\paper_repo\\NIST_sp800-22\\sts-2.1.2")
binfile = sts.write_binary_file(rand_bits, "mydata.bin")
stdout, stderr, report = sts.run_tests("mydata.bin", seq_length=1000000, num_seq=1)

print("NIST STS Finished.\n")
print(report[:1000])  # print first 1000 chars of report


FileNotFoundError: sts.exe not found at ./NIST_sp800-22/nist-sts/sts.exe